In [1]:
!pip install -q kagglehub librosa soundfile xgboost joblib

In [2]:
import kagglehub

path = kagglehub.dataset_download(
    "mohammedabdeldayem/the-fake-or-real-dataset"
)

print(path)

Using Colab cache for faster access to the 'the-fake-or-real-dataset' dataset.
/kaggle/input/the-fake-or-real-dataset


In [3]:
import os
import random

TRAIN_REAL = os.path.join(
    path,
    "for-norm",
    "for-norm",
    "training",
    "real"
)

TRAIN_FAKE = os.path.join(
    path,
    "for-norm",
    "for-norm",
    "training",
    "fake"
)

In [4]:
real_files = [
    os.path.join(TRAIN_REAL,f)
    for f in os.listdir(TRAIN_REAL)
    if f.endswith(".wav")
]

fake_files = [
    os.path.join(TRAIN_FAKE,f)
    for f in os.listdir(TRAIN_FAKE)
    if f.endswith(".wav")
]

print(len(real_files))
print(len(fake_files))

26941
26927


In [5]:
random.seed(42)

N = 5000

real_files = random.sample(
    real_files,
    N
)

fake_files = random.sample(
    fake_files,
    N
)

print(
    len(real_files),
    len(fake_files)
)

5000 5000


In [6]:
import librosa
import numpy as np

In [7]:
def extract_features(
    file_path,
    sr=16000,
    duration=2.0
):

    y, sr = librosa.load(
        file_path,
        sr=sr,
        duration=duration
    )

    target_len = int(
        sr * duration
    )

    if len(y) < target_len:

        y = np.pad(
            y,
            (0,target_len-len(y))
        )

    else:

        y = y[:target_len]

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=20
    )

    delta = librosa.feature.delta(
        mfcc
    )

    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr
    )

    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr
    )

    tonnetz = librosa.feature.tonnetz(
        y=librosa.effects.harmonic(y),
        sr=sr
    )

    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr
    )

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr
    )

    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=sr
    )

    zcr = librosa.feature.zero_crossing_rate(
        y
    )

    rms = librosa.feature.rms(
        y=y
    )

    features = []

    for feat in [

        mfcc,
        delta,
        chroma,
        contrast,
        tonnetz,
        centroid,
        bandwidth,
        rolloff,
        zcr,
        rms

    ]:

        features.append(
            feat.mean()
        )

        features.append(
            feat.std()
        )

        if feat.shape[0] > 1:

            features.extend(
                feat.mean(axis=1)
            )

            features.extend(
                feat.std(axis=1)
            )

    return np.array(
        features,
        dtype=np.float32
    )

In [8]:
from joblib import Parallel
from joblib import delayed

In [9]:
def process_file(
    path,
    label
):

    try:

        return (
            extract_features(path),
            label
        )

    except:

        return None

In [10]:
tasks = []

for fp in real_files:

    tasks.append(
        (fp,0)
    )

for fp in fake_files:

    tasks.append(
        (fp,1)
    )

In [11]:
results = Parallel(
    n_jobs=-1,
    verbose=5
)(
    delayed(process_file)(
        fp,
        label
    )
    for fp,label in tasks
)

results = [
    r for r in results
    if r is not None
]

X = np.array(
    [r[0] for r in results]
)

y = np.array(
    [r[1] for r in results]
)

print(X.shape)
print(y.shape)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:   39.1s
[Parallel(n_jobs=-1)]: Done  68 tasks      | elapsed:   47.0s
[Parallel(n_jobs=-1)]: Done 158 tasks      | elapsed:  1.0min
[Parallel(n_jobs=-1)]: Done 284 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 446 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 644 tasks      | elapsed:  2.3min
[Parallel(n_jobs=-1)]: Done 878 tasks      | elapsed:  3.0min
[Parallel(n_jobs=-1)]: Done 1148 tasks      | elapsed:  3.7min
[Parallel(n_jobs=-1)]: Done 1454 tasks      | elapsed:  4.6min
[Parallel(n_jobs=-1)]: Done 1796 tasks      | elapsed:  5.5min
[Parallel(n_jobs=-1)]: Done 2174 tasks      | elapsed:  6.4min
[Parallel(n_jobs=-1)]: Done 2588 tasks      | elapsed:  7.5min
[Parallel(n_jobs=-1)]: Done 3038 tasks      | elapsed:  8.7min
[Parallel(n_jobs=-1)]: Done 3524 tasks      | elapsed:  9.9min
[Parallel(n_jobs=-1)]: Done 4046 tasks      | ela

(10000, 150)
(10000,)


[Parallel(n_jobs=-1)]: Done 10000 out of 10000 | elapsed: 26.3min finished


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.15,

    stratify=y,

    random_state=42
)

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_test = scaler.transform(
    X_test
)

In [14]:
from sklearn.ensemble import RandomForestClassifier

In [15]:
rf = RandomForestClassifier(

    n_estimators=600,

    max_depth=30,

    min_samples_split=4,

    min_samples_leaf=2,

    n_jobs=-1,

    random_state=42
)

In [16]:
rf.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=30, min_samples_leaf=2, min_samples_split=4,
                       n_estimators=600, n_jobs=-1, random_state=42)

In [17]:
print(
    "Train Accuracy:",
    rf.score(
        X_train,
        y_train
    )
)

print(
    "Test Accuracy:",
    rf.score(
        X_test,
        y_test
    )
)

Train Accuracy: 1.0
Test Accuracy: 0.9746666666666667


In [18]:
from sklearn.metrics import (

    accuracy_score,

    f1_score,

    confusion_matrix,

    classification_report
)

In [19]:
preds = rf.predict(
    X_test
)

probs = rf.predict_proba(
    X_test
)[:,1]

In [20]:
acc = accuracy_score(
    y_test,
    preds
)

f1 = f1_score(
    y_test,
    preds
)

print("Accuracy =",acc)

print("F1 =",f1)

Accuracy = 0.9746666666666667
F1 = 0.975


In [21]:
cm = confusion_matrix(
    y_test,
    preds
)

print(cm)

[[721  29]
 [  9 741]]


In [22]:
print(

    classification_report(
        y_test,
        preds,
        target_names=[
            "Genuine",
            "Deepfake"
        ]
    )
)

              precision    recall  f1-score   support

     Genuine       0.99      0.96      0.97       750
    Deepfake       0.96      0.99      0.97       750

    accuracy                           0.97      1500
   macro avg       0.98      0.97      0.97      1500
weighted avg       0.98      0.97      0.97      1500



In [23]:
from sklearn.metrics import roc_curve

In [24]:
fpr, tpr, _ = roc_curve(
    y_test,
    probs
)

fnr = 1 - tpr

eer_idx = np.nanargmin(
    np.abs(
        fpr - fnr
    )
)

eer = (
    fpr[eer_idx]
    +
    fnr[eer_idx]
) / 2

print(
    "EER =",
    eer
)

EER = 0.02933333333333333


In [25]:
import joblib

joblib.dump(
    rf,
    "model.pkl"
)

joblib.dump(
    scaler,
    "scaler.pkl"
)

print("saved")

saved


In [29]:
from google.colab import files

files.download("model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>